# 03 — CNN đầu vào ảnh xám

Ảnh RGB224 được chuyển xám một kênh trước khi chia255. Mạng giữ ba khối và các tham số huấn luyện như mốc RGB, không augmentation.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA bằng các mức thư mục giới hạn.
from pathlib import Path
import os, sys

kaggle_input = Path('/kaggle/input')
mounts = []
if kaggle_input.is_dir():
    mounts = [p for pattern in ('datasets/*/*', '*')
              for p in kaggle_input.glob(pattern) if p.is_dir()]
project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *mounts,
                       *(child for mount in mounts for child in mount.glob('*') if child.is_dir())])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/CNN/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError('Gắn đúng một Dataset code có src/CNN/pipeline.py; '
                            f'có thể đặt CS231_PROJECT_ROOT. Tìm thấy: {project_matches}')
PROJECT_ROOT = project_matches[0]
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [PROJECT_ROOT / 'DATA', *(p / 'DATA' for p in mounts), *mounts])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir() and (p / 'dev/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError('Gắn đúng Dataset DATA có train/dev; '
                            f'có thể đặt CS231_DATA_ROOT. Tìm thấy: {data_matches}')
DATA_ROOT = data_matches[0]
result_hint = os.environ.get('CS231_CNN_RESULTS_ROOT')
RESULTS_ROOT = (Path(result_hint) if result_hint else
                Path('/kaggle/working/CS231_CNN/results_224') if Path('/kaggle/working').is_dir()
                else PROJECT_ROOT / 'src/CNN/results_224').resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['CS231_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['CS231_DATA_ROOT'] = str(DATA_ROOT)
print('Code:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
from src.CNN.runtime import restore_previous_output
restore_previous_output(RESULTS_ROOT, '02', 'candidates/C01_RGB/run_complete.json')


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image, Markdown
from src.CNN.data import CLASSES, assert_protocol
from src.CNN.runtime import mark_stage
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## 1. Đối chiếu protocol và kiến trúc

So C02 với C01 để khảo sát việc bỏ thông tin màu. Conv đầu đổi từ3 sang1 kênh đầu vào, nên số tham số và các trọng số khởi tạo cũng có thể khác; không coi là kiểm định cô lập hoàn toàn vai trò màu.


In [ ]:
from src.CNN.model import build_model, architecture_table
from src.CNN.pipeline import train_candidate
p = assert_protocol(RESULTS_ROOT, PROJECT_ROOT)
CANDIDATE_ID = 'C02_GRAY'
config = next(c for c in p['candidates'] if c['candidate_id'] == CANDIDATE_ID)
display(pd.DataFrame([config]))
display(pd.DataFrame(architecture_table(config)))
print(build_model(config))


## 2. Huấn luyện và chọn checkpoint trên dev

CrossEntropyLoss nhận logits và chỉ số lớp. AdamW cập nhật trọng số bằng train;
dev được chấm toàn bộ sau mỗi epoch bằng hàm metric chung, sau khi đổi về VND.
Learning rate giảm cosine theo epoch. CUDA dùng AMP và clip gradient norm5.
Patience6 theo Macro-F1 rồi MAE; best.pt giữ epoch thắng, last.pt phục vụ resume.
Lượt đã hoàn tất sẽ được kiểm tra hash và đọc lại khi chạy lại notebook.


In [ ]:
history, summary = train_candidate(CANDIDATE_ID, DATA_ROOT, RESULTS_ROOT, PROJECT_ROOT)
display(history)
display(pd.DataFrame([{k: v for k, v in summary.items() if k != 'environment'}]))
print('Môi trường:', summary['environment'])


## 3. Đọc learning curve và bảng dev

Train loss/Accuracy được tích lũy trong quá trình cập nhật trọng số, có dropout và
augmentation nếu bật. Dev dùng eval với dropout tắt, không augmentation.
Vì cách tính khác nhau, khoảng cách hai đường chỉ là dấu hiệu mô tả;
nhận xét về overfitting cần đọc cả xu hướng loss, Macro-F1 và MAE qua epoch.


In [ ]:
RUN = RESULTS_ROOT / 'candidates' / CANDIDATE_ID
display(Image(filename=str(RUN / 'learning_curves.png')))
display(pd.read_csv(RUN / 'per_class_dev.csv'))
display(Image(filename=str(RUN / 'confusion_matrix_dev.png')))


## 4. Giữ nguyên đầu ra của lượt chạy

Kết quả gồm lịch sử, dự đoán dev, metric, kiến trúc và hai checkpoint.
Các file này đi cùng manifest/protocol trong output để bước sau so sánh cấu hình.
Chưa đánh giá test hoặc kết luận cấu hình thắng trước khi đủ bốn thí nghiệm.


In [ ]:
mark_stage(RESULTS_ROOT, '03')
